In [1]:
import os
import torch
import torch.nn.functional as F
import sys
sys.path.insert(0, "/home/ishan/Desktop/Ishans_workspace/Trans_ReID/TransReID")

from model.backbones import TransReID
from datasets import make_dataloader
from config import cfg
from utils.metrics import R1_mAP_eval

/home/ishan/miniconda3/envs/TransReID/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Create datalaoder for veri-776
cfg.merge_from_file("configs/VeRi/vit_transreid_stride.yml")
cfg.freeze()
train_loader, train_loader_normal, val_loader, num_query, num_classes, cam_num, view_num = make_dataloader(cfg)

Train directory - /home/ishan/Desktop/Ishans_workspace/datasets/VeRI-776/image_train
{0, 1, 2, 3, 4, 5, 6, 7} view_container
63 samples without viewpoint annotations
{0, 1, 2, 3, 4, 5, 6, 7} view_container
0 samples without viewpoint annotations
{0, 1, 2, 3, 4, 5, 6, 7} view_container
0 samples without viewpoint annotations
=> VeRi-776 loaded
Dataset statistics:
  ----------------------------------------
  subset   | # ids | # images | # cameras
  ----------------------------------------
  train    |   576 |    37715 |        20
  query    |   200 |     1678 |        19
  gallery  |   200 |    11579 |        19
  ----------------------------------------


In [3]:
# Create dataloader for market1501
cfg.merge_from_file("configs/Market/vit_transreid_stride.yml")
cfg.freeze()
train_loader, train_loader_normal, val_loader, num_query, num_classes, cam_num, view_num = make_dataloader(cfg)

=> Market1501 loaded
Dataset statistics:
  ----------------------------------------
  subset   | # ids | # images | # cameras
  ----------------------------------------
  train    |   751 |    12936 |         6
  query    |   750 |     3368 |         6
  gallery  |   751 |    15913 |         6
  ----------------------------------------


In [2]:
model = TransReID(
    img_size=(256, 128), 
    stride_size=16, 
    drop_rate=0.0, 
    attn_drop_rate=0.0, 
    drop_path_rate=0.1, 
    camera=0, 
    view=0, 
    local_feature=False, 
    sie_xishu=1.5,
    qkv_bias=True
)

# load weights
weights = "/home/ishan/Desktop/Ishans_workspace/pretrained_weights/jx_vit_base_p16_224-80ecf9dd.pth"
weight_dict = torch.load(weights, map_location='cpu')
model.load_param(weights)

using stride: 16, and patch number is num_y16 * num_x8
using drop_out rate is : 0.0
using attn_drop_out rate is : 0.0
using drop_path rate is : 0.1


/tmp/ipykernel_101873/3566861885.py:16: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  weight_dict = torch.load(weights, map_location='cpu')
/home/ishan/Desktop/Ishans_worksp

Resized position embedding from size:torch.Size([1, 197, 768]) to size: torch.Size([1, 129, 768]) with height:16 width: 8


In [ ]:
def resize_patch_ViT(
    target_image_size: tuple[int],
    image_net_weights: str,
    patch_size: int, 
    stride: int,
    save_path: str,
    return_model: bool = True,
    save_weights: bool = False,
):

    imagenet_sd = torch.load(
        image_net_weights,
        map_location='cpu'
    )

    old_pos_embed = imagenet_sd['pos_embed']
    old_kernel = imagenet_sd['patch_embed.proj.weight']
    old_num_y = old_num_x = 14

    model_p_new = TransReID(
        img_size=target_image_size,
        patch_size=patch_size,
        stride_size=stride,
        drop_rate=0.0,
        attn_drop_rate=0.0,
        drop_path_rate=0.1,
        camera=0,
        view=0,
        local_feature=False,
        sie_xishu=1.5,
        qkv_bias=True
    )

    new_num_y, new_num_x = model_p_new.patch_embed.num_y, model_p_new.patch_embed.num_x
    print(f"patch grid: {old_num_y}x{old_num_x} ({old_num_y*old_num_x} tokens) -> {new_num_y}x{new_num_x} ({new_num_y*new_num_x} tokens)")
    new_sd = {}

    # 2. Resize the conv kernel: bicubic-downsample each [768, 3, 16, 16] filter to [768, 3, 12, 12],
    #    then rescale per output channel so the kernel's L2 norm (and hence activation magnitude
    #    the rest of the pretrained network expects) is preserved.
    new_kernel = F.interpolate(old_kernel, size=(patch_size, patch_size), mode='bicubic', align_corners=False)
    old_norm = old_kernel.flatten(1).norm(dim=1)
    new_norm = new_kernel.flatten(1).norm(dim=1)
    scale = (old_norm / new_norm.clamp_min(1e-8)).view(-1, 1, 1, 1)
    new_kernel = new_kernel * scale

    new_sd['patch_embed.proj.weight'] = new_kernel
    new_sd['patch_embed.proj.bias'] = imagenet_sd['patch_embed.proj.bias']

    # 2. Resize pos_embed ONCE, directly from the pristine 14x14 source grid to the real target grid.
    cls_pos, patch_pos = old_pos_embed[:, :1], old_pos_embed[:, 1:]
    patch_pos = patch_pos.reshape(1, old_num_y, old_num_x, -1).permute(0, 3, 1, 2)
    patch_pos = F.interpolate(patch_pos, size=(new_num_y, new_num_x), mode='bicubic', align_corners=False)
    patch_pos = patch_pos.permute(0, 2, 3, 1).reshape(1, new_num_y * new_num_x, -1)
    new_sd['pos_embed'] = torch.cat([cls_pos, patch_pos], dim=1)

    # 3. Everything else (cls_token, blocks.*, norm.*) is patch-size-agnostic -> copy straight from the
    #    pristine checkpoint. Skip 'head.*' (ImageNet's 1000-class head, unused for ReID) same as load_param does.
    for k, v in imagenet_sd.items():
        if k in new_sd or 'head' in k or 'dist' in k:
            continue
        new_sd[k] = v

    result = model_p_new.load_state_dict(new_sd, strict=False)
    print(result)  # expect <All keys matched successfully>

    if save_weights:
        torch.save(
            model_p_new.state_dict(),
                save_path,
        )

    if return_model:
        return model_p_new

In [22]:
model = resize_patch_ViT(
    target_image_size=(256, 256), 
    patch_size=8, 
    stride=8, 
    save_path="/home/ishan/Desktop/Ishans_workspace/pretrained_weights/imagenet_vit8.pth",
    image_net_weights="/home/ishan/Desktop/Ishans_workspace/pretrained_weights/jx_vit_base_p16_224-80ecf9dd.pth"
)

/tmp/ipykernel_101873/288028891.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  imagenet_sd = torch.load(


using stride: 8, and patch number is num_y32 * num_x32
using drop_out rate is : 0.0
using attn_drop_out rate is : 0.0
using drop_path rate is : 0.1
patch grid: 14x14 (196 tokens) -> 32x32 (1024 tokens)
_IncompatibleKeys(missing_keys=['fc.weight', 'fc.bias'], unexpected_keys=[])


In [10]:
old_kernel.shape

torch.Size([768, 3, 16, 16])

In [5]:
torch.save(
    model_p12.state_dict(),
    f"/home/ishan/Desktop/Ishans_workspace/pretrained_weights/imagenet_vit12.pth")

In [5]:
# Inference loop 
from tqdm.auto import tqdm

device = torch.device("cuda")
model = model.to(device)
model.eval()

img_path_list = []
evaluator = R1_mAP_eval(num_query=num_query, max_rank=50, feat_norm=cfg.TEST.FEAT_NORM)
evaluator.reset()

with torch.no_grad():
    for n_iter, (img, pid, camid, camids, target_view, imgpath) in tqdm(
        enumerate(val_loader),
        total=len(val_loader),
        desc="Evaluating on test set..."
    ):

        img = img.to(device)
        camids = camids.to(device)
        target_view = target_view.to(device)
        feat = model(img, cam_label=camids, view_label=target_view)
        evaluator.update((feat, pid, camid))
        img_path_list.extend(imgpath)

Evaluating on test set...: 100%|██████████| 38/38 [02:51<00:00,  4.51s/it]


In [6]:
cmc, mAP, _, _, _, _, _ = evaluator.compute()

The test feature is normalized
=> Computing DistMat with euclidean_distance


/home/ishan/Desktop/Ishans_workspace/Trans_ReID/TransReID/utils/metrics.py:12: UserWarning: This overload of addmm_ is deprecated:
	addmm_(Number beta, Number alpha, Tensor mat1, Tensor mat2)
Consider using one of the following signatures instead:
	addmm_(Tensor mat1, Tensor mat2, *, Number beta = 1, Number alpha = 1) (Triggered internally at ../torch/csrc/utils/python_arg_parser.cpp:1642.)
  dist_mat.addmm_(1, -2, qf, gf.t())


In [7]:
print(f"Rank-1: {cmc[0]:.4f}")
print(f"Rank-5: {cmc[4]:.4f}")
print(f"Rank-10: {cmc[9]:.4f}")
print(f"mAP: {mAP:.4f}")

Rank-1: 0.9469
Rank-5: 0.9825
Rank-10: 0.9920
mAP: 0.8718
